# 02 — Forward pass, hidden states, and why hybrid backbones can't do block-causal packing

**Recap from notebook 1:** the language backbone is 24 layers, pattern `[DeltaNet, DeltaNet, DeltaNet, Attention]` × 6. `model.visual` and `model.language_model` are separate subtrees; `lm_head` is tied to the input embedding.

**Goal of this notebook:** any2jev's README asserted, as a one-line claim, that "hybrid backbones (Qwen3.5, linear attention) fall back to one causal row per question" because they "cannot take the block-causal mask." We're not going to trust that sentence — we're going to find out *ourselves*, from the actual forward pass and the actual source code of `Qwen3_5GatedDeltaNet`, whether that's true and *why*.

To get there we need to understand two things concretely:
1. What a forward pass actually returns (hidden states per layer, and a cache object for incremental decoding).
2. What a Gated Attention layer caches vs. what a Gated DeltaNet layer caches — these turn out to be fundamentally different *kinds* of objects, not just different shapes.

This notebook assumes `processor` and `model` are already loaded exactly as in notebook 1 — rerun that notebook's loading cell first (or just continue in the same kernel/session if it's still alive).

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "Qwen/Qwen3.5-0.8B"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
model.eval()
print("loaded.")

loaded.


## Step 1: a plain text forward pass

We'll start text-only (no image) to keep the first pass simple — the vision tower is a separate question for later. We use the chat template (the exact formatting string this model was trained on) rather than hand-writing the prompt, since getting delimiters/roles wrong changes the model's behavior in ways that have nothing to do with what we're studying here.

`output_hidden_states=True` asks the model to return the hidden state tensor *after every layer*, not just the final one — normally discarded to save memory, but exactly what we want to inspect. `use_cache=True` asks it to return whatever each layer needs to remember in order to continue generating token-by-token later (relevant in a moment).

In [2]:
messages = [{"role": "user", "content": [{"type": "text", "text": "The customer says their payout failed three days in a row. Is this urgent?"}]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(model.device)

print("input_ids shape:", inputs["input_ids"].shape)
print(decoded := processor.decode(inputs["input_ids"][0]))

input_ids shape: torch.Size([1, 28])
<|im_start|>user
The customer says their payout failed three days in a row. Is this urgent?<|im_end|>
<|im_start|>assistant
<think>

</think>




**Look at the decoded string above.** This is the chat template's actual output — the literal text the model sees, special tokens and all. Worth noting what role markers / delimiters appear, since those are exactly the kind of tokens any2jev's `<state>`/`<q>`/`<opt>`/`<decide>` scheme adds *more* of — we'll be designing our own set of these later.

Now the actual forward pass:

In [3]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True, use_cache=True)

print("logits shape:", outputs.logits.shape)
print("number of hidden_states tensors:", len(outputs.hidden_states))
print("shape of each:", outputs.hidden_states[0].shape)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


logits shape: torch.Size([1, 28, 248320])
number of hidden_states tensors: 25
shape of each: torch.Size([1, 28, 1024])


**Answer before moving on:** the model has 24 layers, but `len(outputs.hidden_states)` is probably not 24. Why? (Hint: one of these tensors is the *input* to the stack — the raw token embeddings before any layer has touched them — not the output of a layer.) Confirm your answer by checking `torch.equal(outputs.hidden_states[0], model.get_input_embeddings()(inputs["input_ids"]))` roughly holds (dtype/device permitting).

## Step 2: what does each layer type actually cache?

This is the crux of the notebook. `outputs.past_key_values` is the thing a model hands back so that generating the *next* token doesn't require reprocessing every previous token from scratch. For an ordinary transformer, this is simple: every layer caches a Key tensor and a Value tensor, one per past position — literally a growing list of vectors, one per token seen so far.

But we have two different layer types. Let's look at what each one actually stored, indexing by the layer numbers we already know from notebook 1 (`layer 0` = Gated DeltaNet, `layer 3` = Gated Attention).

In [4]:
cache = outputs.past_key_values
print("type of the whole cache object:", type(cache))
print(dir(cache))

type of the whole cache object: <class 'transformers.cache_utils.DynamicCache'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'activate_past_recording', 'batch_repeat_interleave', 'batch_select_indices', 'batch_size', 'crop', 'early_initialization', 'get_mask_sizes', 'get_max_cache_shape', 'get_max_length', 'get_query_offset', 'get_seq_length', 'has_previous_state', 'is_compileable', 'is_croppable', 'is_initialized', 'is_linear', 'is_sliding', 'layer_class_to_replicate', 'layers', 'max_batch_size', 'max_cache_len', 'offload', 'offloading', 'prefetch', 'reorder_cache', 'reset', 'update', 'update_conv_state', 'up

In [7]:
print(cache)

DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer])


In [10]:
vars(cache.layers[0])

{'number_of_states': 1,
 'conv_states': {0: tensor([[[-0.6133,  0.2598,  0.1191,  0.2598],
           [ 0.5000, -0.0505,  0.4902, -0.0505],
           [-0.0588, -0.1748,  0.4805, -0.1748],
           ...,
           [-1.2344,  0.5586, -1.1719,  0.5586],
           [-1.2969,  0.4609, -1.1172,  0.4609],
           [-2.2969,  0.5625, -1.2734,  0.5625]]], device='cuda:0',
         dtype=torch.bfloat16)},
 'recurrent_states': {0: tensor([[[[-3.2443e-04,  3.6321e-04,  4.3094e-06,  ...,  2.3863e-05,
             -2.0915e-04, -2.8498e-03],
            [-3.2981e-04,  3.6817e-04,  8.1193e-06,  ...,  2.5789e-05,
             -2.1719e-04, -2.9293e-03],
            [ 6.3558e-04, -7.1222e-04, -7.4842e-06,  ..., -4.5917e-05,
              4.0912e-04,  5.5695e-03],
            ...,
            [ 6.3704e-04, -7.1802e-04, -5.4745e-06,  ..., -4.1213e-05,
              4.1256e-04,  5.5170e-03],
            [-4.4948e-04,  5.0489e-04,  1.9629e-06,  ...,  3.0822e-05,
             -2.8562e-04, -3.9066e-03],
 

In [11]:
vars(cache.layers[3])

{'keys': tensor([[[[-0.0090,  0.2070,  0.3105,  ...,  1.0938,  1.9766, -3.3750],
           [ 0.4629,  0.4316,  0.2812,  ...,  0.1523,  1.4141, -1.8516],
           [-0.0713, -0.0430, -0.1973,  ...,  0.2559,  1.5625, -1.1094],
           ...,
           [-0.2891, -0.0576,  0.2324,  ...,  0.0238,  0.7578, -1.1953],
           [ 0.9727, -1.0312, -1.5078,  ...,  1.0938,  0.9961, -1.3047],
           [-0.6211, -0.1562, -0.6953,  ..., -0.3633,  0.6758, -0.8516]],
 
          [[ 0.1270,  0.0723, -0.0067,  ...,  0.3457, -0.6953,  0.9805],
           [-0.2363, -0.1309,  0.2793,  ...,  1.9766,  0.4141, -0.6719],
           [-0.0840,  0.0967, -0.3242,  ...,  1.5391,  0.5117, -0.4121],
           ...,
           [ 0.1611, -0.4688,  0.7070,  ...,  1.1875, -0.6289,  1.5078],
           [ 1.1875, -1.1562,  2.4062,  ...,  1.7031,  1.1562,  0.7383],
           [-0.3574, -0.6172,  1.5781,  ...,  0.8398, -0.4082,  1.5938]]]],
        device='cuda:0', dtype=torch.bfloat16),
 'values': tensor([[[[-0.0801,

In [13]:
print("DeltaNet layer (0) conv_states shape:", cache.layers[0].conv_states[0].shape)
print("DeltaNet layer (0) recurrent_states shape:", cache.layers[0].recurrent_states[0].shape)
print("Attention layer (3) keys shape:", cache.layers[3].keys.shape)
print("Attention layer (3) values shape:", cache.layers[3].values.shape)

DeltaNet layer (0) conv_states shape: torch.Size([1, 6144, 4])
DeltaNet layer (0) recurrent_states shape: torch.Size([1, 16, 128, 128])
Attention layer (3) keys shape: torch.Size([1, 2, 28, 256])
Attention layer (3) values shape: torch.Size([1, 2, 28, 256])


In [14]:
print(model.config.text_config if hasattr(model.config, "text_config") else model.config)

Qwen3_5TextConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_output_gate": true,
  "bos_token_id": null,
  "dtype": "bfloat16",
  "eos_token_id": 248044,
  "full_attention_interval": 4,
  "head_dim": 256,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3584,
  "layer_types": [
    "linear_attention",
    "linear_attention",
    "linear_attention",
    "full_attention",
    "linear_attention",
    "linear_attention",
    "linear_attention",
    "full_attention",
    "linear_attention",
    "linear_attention",
    "linear_attention",
    "full_attention",
    "linear_attention",
    "linear_attention",
    "linear_attention",
    "full_attention",
    "linear_attention",
    "linear_attention",
    "linear_attention",
    "full_attention",
    "linear_attention",
    "linear_attention",
    "linear_attention",
    "full_attention"
  ],
  "linear_conv_kernel_dim": 4,
  "linear_key_head_dim": 128,
  "linear_num_key_he

Skim the attribute list. Look for anything that sounds like it holds per-layer state — common names in `transformers` cache classes are things like `key_cache` / `value_cache` (lists indexed by layer), or newer unified designs might use a single `layers` list of per-layer cache objects. Try a few things in the next cell until one works — this is deliberately exploratory, there's no single right first guess.

In [5]:
# Try things like:
# cache.layers[0], cache.layers[3]
# or cache.key_cache[0].shape vs cache.key_cache[3].shape
# Print whatever you find for layer 0 (DeltaNet) and layer 3 (Attention) side by side.

layer0_state = cache.layers[0]
layer3_state = cache.layers[3]
print("layer 0 (DeltaNet) cache entry:", type(layer0_state))
print(layer0_state)
print()
print("layer 3 (Attention) cache entry:", type(layer3_state))
print(layer3_state)

layer 0 (DeltaNet) cache entry: <class 'transformers.cache_utils.LinearAttentionLayer'>
LinearAttentionLayer

layer 3 (Attention) cache entry: <class 'transformers.cache_utils.DynamicLayer'>
DynamicLayer


**The key question to answer from the output above:** does the Attention layer's cache entry *grow with sequence length* (shape has a dimension equal to, or related to, your `input_ids` length), while the DeltaNet layer's entry has a shape that's **fixed size regardless of how many tokens you've fed it** (e.g. tied to head dimension, not sequence length)?

If so, you've just found, with your own eyes, the actual mechanical reason any2jev's hybrid-backbone claim is true: a normal KV cache is a *memory of every position* — you can later choose to let a new query attend to some of those remembered positions and not others, which is exactly what a block-causal attention mask is: a per-position include/exclude list. A **fixed-size recurrent state** has already irreversibly mixed everything it's seen into one blob of numbers of constant size — there is no "list of positions" left to selectively mask, because the layer never kept one.

## Step 3: confirm it from the source code, not just the cache shape

Shapes are suggestive but let's be rigorous: does `Qwen3_5GatedDeltaNet.forward` even *accept* an `attention_mask` argument the way `Qwen3_5Attention.forward` does? If its signature doesn't take one (or silently ignores it), that's the strongest possible evidence — it's not a performance limitation, it's structurally impossible to pass a custom block-causal mask into that layer type without rewriting it.

In [6]:
import inspect

# Find the actual classes from the layer list in notebook 1.
layers = model.model.language_model.layers
delta_layer = layers[0]
attn_layer = layers[3]

delta_mixer = [m for n, m in delta_layer.named_children()][0]
attn_mixer = [m for n, m in attn_layer.named_children()][0]

print("=== DeltaNet mixer forward signature ===")
print(inspect.signature(delta_mixer.forward))
print()
print("=== Attention mixer forward signature ===")
print(inspect.signature(attn_mixer.forward))

=== DeltaNet mixer forward signature ===
(hidden_states: torch.Tensor, cache_params: transformers.cache_utils.Cache | None = None, attention_mask: torch.Tensor | None = None, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs])

=== Attention mixer forward signature ===
(hidden_states: torch.Tensor, position_embeddings: tuple[torch.Tensor, torch.Tensor], attention_mask: torch.Tensor | None, past_key_values: transformers.cache_utils.Cache | None = None, **kwargs: Unpack[transformers.modeling_flash_attention_utils.FlashAttentionKwargs]) -> tuple[torch.Tensor, torch.Tensor | None]


In [16]:
list(delta_layer.named_children())

[('linear_attn',
  Qwen3_5GatedDeltaNet(
    (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
    (norm): Qwen3_5RMSNormGated()
    (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (in_proj_qkv): Linear(in_features=1024, out_features=6144, bias=False)
    (in_proj_z): Linear(in_features=1024, out_features=2048, bias=False)
    (in_proj_b): Linear(in_features=1024, out_features=16, bias=False)
    (in_proj_a): Linear(in_features=1024, out_features=16, bias=False)
  )),
 ('mlp',
  Qwen3_5MLP(
    (gate_proj): Linear(in_features=1024, out_features=3584, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3584, bias=False)
    (down_proj): Linear(in_features=3584, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )),
 ('input_layernorm', Qwen3_5RMSNorm((1024,), eps=1e-06)),
 ('post_attention_layernorm', Qwen3_5RMSNorm((1024,), eps=1e-06))]

In [17]:
list(attn_layer.named_children())

[('self_attn',
  Qwen3_5Attention(
    (q_proj): Linear(in_features=1024, out_features=4096, bias=False)
    (k_proj): Linear(in_features=1024, out_features=512, bias=False)
    (v_proj): Linear(in_features=1024, out_features=512, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3_5RMSNorm((256,), eps=1e-06)
    (k_norm): Qwen3_5RMSNorm((256,), eps=1e-06)
  )),
 ('mlp',
  Qwen3_5MLP(
    (gate_proj): Linear(in_features=1024, out_features=3584, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3584, bias=False)
    (down_proj): Linear(in_features=3584, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )),
 ('input_layernorm', Qwen3_5RMSNorm((1024,), eps=1e-06)),
 ('post_attention_layernorm', Qwen3_5RMSNorm((1024,), eps=1e-06))]

In [18]:
import inspect
print("=== DeltaNet mixer source ===")
print(inspect.getsource(type(delta_mixer).forward))

=== DeltaNet mixer source ===
    @force_accelerate_hooks("conv1d")
    def forward(
        self,
        hidden_states: torch.Tensor,
        cache_params: Cache | None = None,
        attention_mask: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ):
        hidden_states = apply_mask_to_padding_states(hidden_states, attention_mask)

        # Set up dimensions for reshapes later
        batch_size, seq_len, _ = hidden_states.shape
        use_precomputed_states = cache_params is not None and cache_params.has_previous_state(
            self.layer_idx, state_idx=0
        )

        mixed_qkv = self.in_proj_qkv(hidden_states)
        mixed_qkv = mixed_qkv.transpose(1, 2)

        z = self.in_proj_z(hidden_states)
        z = z.reshape(batch_size, seq_len, -1, self.head_v_dim)

        b = self.in_proj_b(hidden_states)
        a = self.in_proj_a(hidden_states)

        if use_precomputed_states and seq_len == 1 and not cache_params.layers[self.layer_idx]

In [19]:
print("=== Attention mixer source ===")
print(inspect.getsource(type(attn_mixer).forward))

=== Attention mixer source ===
    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: torch.Tensor | None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[FlashAttentionKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states, gate = torch.chunk(
            self.q_proj(hidden_states).view(*input_shape, -1, self.head_dim * 2), 2, dim=-1
        )
        gate = gate.reshape(*input_shape, -1)

        query_states = self.q_norm(query_states.view(hidden_shape)).transpose(1, 2)
        key_states = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotar

**Write your conclusion as a comment below:** does `attention_mask` (or an equivalent masking argument) appear in both signatures, one, or neither? Given what you found, what would "rows mode" (any2jev's fallback for hybrid backbones — running one full causal pass per question, over `state + that one question`, instead of packing all questions into a single masked sequence) actually have to look like as code? You don't need to implement it yet — just describe, in your own words, the loop you'd write: what goes into each forward call, and why you can't avoid redoing the state-prefix computation for every question when using this backbone.

## Wrap-up

You now have, from direct observation rather than documentation:
- What `hidden_states` and `past_key_values` actually are and how many layers' worth of each you get.
- A concrete, shape-level difference between what a Gated Attention layer remembers (grows with sequence length) and what a Gated DeltaNet layer remembers (fixed-size recurrent state).
- Source-level confirmation of whether each layer type can even accept a custom attention mask.

Next notebook: we design the actual jev data format ourselves — what a Choice/Score/Noul question record looks like, what delimiter tokens we need, and how "rows mode" gets implemented as real code for this specific backbone.